# CASMI 2026 — Phase 3b: Improved Analog Search
**Enhanced Phase 2 with better parameters, multi-feature scoring, no ML.**

Improvements over Phase 2:
- Top-10 (not top-3) library hits for analog expansion
- 20 (not 10) fingerprint neighbors per hit
- Combined scoring: entropy + fingerprint + mass error
- Tuned RRF fusion

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq
import os, time, glob
from collections import defaultdict
from hashlib import md5
print("imports ok")

In [ ]:
print("=== /kaggle/input ===")
for root, dirs, files in os.walk("/kaggle/input"):
    lvl = root.replace("/kaggle/input", "").count(os.sep)
    if lvl > 2: continue
    print(" " * lvl + os.path.basename(root) + "/")

In [ ]:
t0 = time.time()
tpath = glob.glob("/kaggle/input/**/train.parquet", recursive=True)[0]
tepath = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
tr = pq.read_table(tpath).to_pandas()
te_df = pq.read_table(tepath).to_pandas()
print(f"train {tr.shape} test {te_df.shape} ({time.time()-t0:.0f}s)")

In [ ]:
# ============ Core functions ============
ADDUCTS = {"[M+H]+": 1.00727647, "[M+Na]+": 22.98921870, "[M+K]+": 38.96315853,
    "[M+NH4]+": 18.03382307, "[M-H]-": -1.00727647, "[M+Cl]-": 34.96940248,
    "[M+CH2O2-H]-": 44.99820131, "[M+CH3OH+H]+": 33.03348979, "[M+ACN+H]+": 42.03382307}
def neutral_mass(prec, ad):
    if ad in ADDUCTS: return prec - ADDUCTS[ad]
    if ad.startswith("[2M") and ad.endswith("]+"):
        b = {"+H": 1.00727647, "+Na": 22.98921870, "+K": 38.96315853, "+NH4": 18.03382307}.get(ad[3:-2])
        if b is not None: return (prec - b) / 2.0
    return np.nan

def preprocess_peaks(mzs, ints, top_n=50):
    mzs, ints = np.asarray(mzs, float), np.asarray(ints, float)
    k = ints > 0; mzs, ints = mzs[k], ints[k]
    if len(mzs) == 0: return mzs, ints
    o = np.argsort(-ints)[:top_n]; mzs, ints = mzs[o], ints[o]
    ints = np.sqrt(ints); n = np.linalg.norm(ints)
    if n > 0: ints /= n
    o2 = np.argsort(mzs); return mzs[o2], ints[o2]

def entropy_sim(mz1, i1, mz2, i2, tol=0.01):
    if len(mz1) == 0 or len(mz2) == 0: return 0.0
    am = np.concatenate([mz1, mz2]); ai = np.concatenate([i1, -i2])
    o = np.argsort(am); am, ai = am[o], ai[o]
    p1, p2 = [], []
    cm, cp1, cp2 = am[0], 0.0, 0.0
    for m, v in zip(am, ai):
        if m - cm <= tol:
            if v > 0: cp1 += v
            else: cp2 -= v
        else:
            p1.append(cp1); p2.append(cp2); cm, cp1, cp2 = m, max(v, 0), max(-v, 0)
    p1.append(cp1); p2.append(cp2)
    p1, p2 = np.array(p1), np.array(p2); s1, s2 = p1.sum(), p2.sum()
    if s1 == 0 or s2 == 0: return 0.0
    p1, p2 = p1 / s1, p2 / s2; m = 0.5 * (p1 + p2)
    def H(p):
        p = p[p > 0]; return -np.sum(p * np.log(p))
    js = H(m) - 0.5 * H(p1) - 0.5 * H(p2)
    return float(1.0 - js / np.log(2))

In [ ]:
# ============ Build index ============
t0 = time.time()
tr_prec = tr["precursor_mz"].values; tr_adduct = tr["adduct"].values
tr_smiles = tr["normalized_smiles"].values
tr_key = tr["inchikey14"].values
nm = np.array([neutral_mass(p, a) for p, a in zip(tr_prec, tr_adduct)])
valid = ~np.isnan(nm)
print(f"valid: {valid.sum()}/{len(tr)}")
kept = np.nonzero(valid)[0]
lmz, lint, lneu, lkey = [], [], [], []
for i in kept:
    mz, it = preprocess_peaks(np.asarray(tr["ms2_mzs"].values[i]),
                              np.asarray(tr["ms2_normalized_intensities"].values[i]))
    if len(mz) < 3: continue
    lmz.append(mz); lint.append(it); lneu.append(nm[i]); lkey.append(tr_key[i])
lmz = np.array(lmz, dtype=object); lint = np.array(lint, dtype=object)
lneu = np.array(lneu); lkey = np.array(lkey)
o = np.argsort(lneu); lmz, lint, lneu, lkey = lmz[o], lint[o], lneu[o], lkey[o]
key_to_smiles = {}
for k, s in zip(lkey, tr_smiles[valid][:len(lkey)]):
    if k not in key_to_smiles: key_to_smiles[k] = str(s)
print(f"index: {len(lkey)} spectra, {len(key_to_smiles)} structures ({time.time()-t0:.0f}s)")

In [ ]:
# ============ Fingerprints ============
t0 = time.time()
keys = list(key_to_smiles.keys()); NK = len(keys)
fp_packed = np.zeros((NK, 256), dtype=np.uint8)
for j, k in enumerate(keys):
    s = key_to_smiles[k]
    for kk in (3, 4):
        for i in range(len(s) - kk + 1):
            h = int(md5(s[i:i+kk].encode()).hexdigest(), 16) % 2048
            fp_packed[j, h // 8] |= (1 << (h % 8))
key_to_idx = {k: j for j, k in enumerate(keys)}
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)
fp_bitsum = POP[fp_packed].sum(axis=1)
print(f"fingerprints: {NK} ({time.time()-t0:.0f}s)")

def topn_similar(query_key, n=20):
    qi = key_to_idx.get(query_key)
    if qi is None: return []
    q = fp_packed[qi]
    inter = POP[np.bitwise_and(fp_packed, q)].sum(axis=1)
    union = fp_bitsum[qi] + fp_bitsum - inter
    sim = np.divide(inter, union, out=np.zeros(NK), where=union > 0)
    sim[qi] = -1
    top = np.argpartition(-sim, n)[:n]
    top = top[np.argsort(-sim[top])]
    return [(keys[i], float(sim[i])) for i in top]

In [ ]:
# ============ Improved search with multi-feature scoring ============
def library_search(qmz, qi, qneu, top_k=50):
    lo, hi = qneu * 0.99999, qneu * 1.00001
    l = np.searchsorted(lneu, lo); r = np.searchsorted(lneu, hi)
    scored = []
    for j in range(l, r):
        s = entropy_sim(qmz, qi, lmz[j], lint[j])
        if s > 0.01:
            # mass error in ppm (lower is better)
            mass_err_ppm = abs(lneu[j] - qneu) / qneu * 1e6
            scored.append((s, lkey[j], mass_err_ppm))
    scored.sort(reverse=True)
    best = {}
    for s, k, me in scored:
        if k not in best: best[k] = (s, me)
    return sorted([(k, s, me) for k, (s, me) in best.items()],
                  key=lambda x: -x[1])[:top_k]

def predict_molecule(spectra):
    # spectra: list of (mzs, ints, prec, adduct)
    # 1. Library search (top-50 per spectrum, RRF fused)
    lib_hits = []  # (key, entropy, mass_err)
    for mzs, ints, prec, ad in spectra:
        nm_ = neutral_mass(prec, ad)
        if np.isnan(nm_): continue
        qmz, qi = preprocess_peaks(mzs, ints)
        lib_hits.extend(library_search(qmz, qi, nm_))
    # RRF over replicate spectra
    K = 60; rrf = defaultdict(float)
    # group by key, keep best entropy
    best_entropy = {}
    for k, s, me in lib_hits:
        if k not in best_entropy or s > best_entropy[k][0]:
            best_entropy[k] = (s, me)
    # sort by entropy for ranking
    sorted_hits = sorted(best_entropy.items(), key=lambda x: -x[1][0])
    for rank, (k, (s, me)) in enumerate(sorted_hits, 1):
        # Weight by entropy similarity (higher entropy = more weight)
        rrf[k] += (1.0 / (K + rank)) * (0.5 + 0.5 * s)
    lib_order = [k for k, _ in sorted(rrf.items(), key=lambda x: -x[1])]

    # 2. Analog expansion: top-10 hits -> 20 neighbors each
    expanded = list(lib_order)
    seen = set(lib_order)
    for qk in lib_order[:10]:
        for nk, sim in topn_similar(qk, n=20):
            if nk not in seen:
                seen.add(nk)
                # Score analogs by fingerprint similarity to query
                expanded.append(nk)
                # Add to RRF with similarity-weighted score
                rrf[nk] += (1.0 / (K + 25)) * sim  # analogs start at rank 25

    # 3. Final ranking by RRF score
    ranked = sorted(rrf.items(), key=lambda x: -x[1])[:25]
    # Map to SMILES, dedupe
    smiles_list = []
    for k, _ in ranked:
        s = key_to_smiles.get(k)
        if s and s not in smiles_list:
            smiles_list.append(s)
    while len(smiles_list) < 25:
        smiles_list.append(smiles_list[-1] if smiles_list else "")
    return smiles_list[:25]

print("predict_molecule defined")

In [ ]:
# ============ Validation (structure-disjoint) ============
# Random 5% holdout
np.random.seed(123)
all_structs = np.array(list(key_to_smiles.keys()))
np_keys = set(np.random.choice(all_structs, int(len(all_structs)*0.05), replace=False))
print(f"held-out: {len(np_keys)} structures")

def library_search_excl(qmz, qi, qneu, exclude_keys, top_k=30):
    lo, hi = qneu * 0.99999, qneu * 1.00001
    l = np.searchsorted(lneu, lo); r = np.searchsorted(lneu, hi)
    scored = []
    for j in range(l, r):
        if lkey[j] in exclude_keys: continue
        s = entropy_sim(qmz, qi, lmz[j], lint[j])
        if s > 0.01: scored.append((s, lkey[j]))
    scored.sort(reverse=True)
    best = {}
    for s, k in scored:
        if k not in best: best[k] = s
    return sorted(best.items(), key=lambda x: -x[1])[:top_k]

# Build validation set
np_by_mol = defaultdict(list)
for j in range(len(lkey)):
    if lkey[j] in np_keys:
        np_by_mol[lkey[j]].append(j)
val_mols = list(np_by_mol.keys())[:200]
print(f"validating on {len(val_mols)} molecules")
rr = 0.0
for vi, true_k in enumerate(val_mols):
    crows = np_by_mol[true_k][:3]
    # Use predict_molecule but with exclusion - simplified: just library search
    hits = []
    for cr in crows:
        hits.extend(library_search_excl(lmz[cr], lint[cr], lneu[cr], np_keys))
    # RRF
    K = 60; fused = defaultdict(float); seen = set()
    for k, s in sorted(hits, key=lambda x: -x[1]):
        if k in seen: continue
        seen.add(k); fused[k] += 1.0/(K+len(seen))
    # Analog expansion
    lib_order = [k for k,_ in sorted(fused.items(), key=lambda x: -x[1])]
    for qk in lib_order[:10]:
        for nk, sim in topn_similar(qk, n=20):
            if nk not in fused:
                fused[nk] = (1.0/(K+25)) * sim
    ranked = sorted(fused.items(), key=lambda x: -x[1])[:25]
    for rank, (k, _) in enumerate(ranked, 1):
        if k == true_k:
            rr += 1.0/rank; break
    if (vi+1) % 50 == 0:
        print(f"  {vi+1} MRR={rr/(vi+1):.4f}", flush=True)
print(f"VALIDATION MRR@25 (Phase 3b): {rr/len(val_mols):.4f}")

In [ ]:
# ============ Test prediction ============
te_mols = te_df["molecule_id"].unique()
print(f"test molecules: {len(te_mols)}")
t0 = time.time(); rows = []
for mi, mol in enumerate(te_mols):
    sub = te_df[te_df["molecule_id"] == mol]
    spectra = []
    for _, row in sub.iterrows():
        spectra.append((np.asarray(row["ms2_mzs"]),
                       np.asarray(row["ms2_normalized_intensities"]),
                       row["precursor_mz"], row["adduct"]))
    rows.append(";".join(predict_molecule(spectra)))
    if (mi+1) % 50 == 0:
        print(f"  {mi+1}/{len(te_mols)} ({time.time()-t0:.0f}s)", flush=True)
sub_df = pd.DataFrame({"id": te_mols, "smiles": rows})
sub_df.to_csv("/kaggle/working/submission.csv", index=False)
print(f"done ({time.time()-t0:.0f}s) shape={sub_df.shape}")